# 07 — تحلیل حساسیت و Claim Registry نهایی

این Notebook فاز دوازدهم (§24 مقایسه‌ی گروه‌ها) و فاز پانزدهم (تحلیل حساسیت، حداقل شش مقایسه) checklist.md را روایت می‌کند و با ارجاع به Claim Registry نهایی می‌رسد.

**بخش ۱:** مقایسه‌ی گروه‌ها (§24) — فراخوانی خروجی `src/temporal_analysis/group_comparison.py`
**بخش ۲:** تحلیل حساسیت (فاز پانزدهم) — فراخوانی خروجی `src/temporal_analysis/sensitivity_analysis.py`
**بخش ۳:** Claim Registry — خلاصه‌ی ادعاهای مستند (`docs/claim_registry.md`)

هشدار داده: این Notebook روی `data/processed/annotated_dataset.parquet` واقعی اجرا می‌شود، اما annotation فقط
**۳.۷٪** پوشش دارد (`docs/handoff_notes_fa.md`) — همه‌ی اعداد این Notebook **مقدماتی/Exploratory** هستند، نه نتیجه‌ی نهایی.

In [1]:
from pathlib import Path
import sys

def find_project_root(start=None):
    here = (start or Path.cwd()).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src" / "temporal_analysis").exists() or (candidate / "config" / "config.yaml").exists():
            return candidate
    raise RuntimeError("Project root not found -- run this notebook from inside the repo.")

ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f"Project root: {ROOT}")

Project root: C:\Users\user\OneDrive\Desktop\hamrahaval\final\media-sentiment-pipeline-starter\media-sentiment-pipeline-starter


In [2]:
import pandas as pd

REAL_INPUT   = ROOT / "data" / "processed" / "annotated_dataset.parquet"
SAMPLE_INPUT = ROOT / "data" / "processed" / "annotated_dataset.sample.parquet"
TABLES_DIR   = ROOT / "outputs" / "tables"

real_data_ready = REAL_INPUT.exists()
input_path = REAL_INPUT if real_data_ready else SAMPLE_INPUT
print({
    "real_annotated_dataset_exists": real_data_ready,
    "using_input": str(input_path.relative_to(ROOT)),
})
if not real_data_ready:
    print("STATUS: pending_annotated_dataset -- run src/annotation/build_annotated_dataset.py first.")

{'real_annotated_dataset_exists': True, 'using_input': 'data\\processed\\annotated_dataset.parquet'}


---
## بخش ۱ — مقایسه‌ی گروه‌ها (§24)

جدول ۴ آزمون دقیقاً طبق checklist.md §24: Chi-square+Cramer's V (توزیع Stance بین پلتفرم/زبان)، Fisher's Exact+Odds Ratio (جدول ۲×۲: support/oppose بین هر جفت پلتفرم)، Mann-Whitney U+rank-biserial (Engagement)، Welch's t-test+Hedges' g (طول متن).

**قاعده:** فقط p<0.05 گزارش نمی‌شود — هر آزمونی که اجرا شده (چه معنادار چه نه) با n، Estimate، CI، Effect size و فرض‌هایش در جدول زیر هست.

In [3]:
gc_path = TABLES_DIR / "group_comparison_results.csv"
if gc_path.exists():
    group_comparison = pd.read_csv(gc_path)
    n_sig = int((group_comparison["p_value"] < 0.05).sum())
    print(f"{len(group_comparison)} test(s) -- {n_sig} with p<0.05 (all {len(group_comparison)} reported regardless)")
    display(group_comparison[[
        "comparison_id", "test", "n", "estimate", "effect_size",
        "effect_size_ci_low", "effect_size_ci_high", "p_value",
    ]])
else:
    print("STATUS: pending -- run `python -m src.temporal_analysis.group_comparison` first.")

12 test(s) -- 8 with p<0.05 (all 12 reported regardless)


,comparison_id,test,n,estimate,effect_size,effect_size_ci_low,effect_size_ci_high,p_value
0,chi_square_stance_platform,chi_square,9405,NaN,0.146827,0.135610,0.160059,1.243065e-82
1,chi_square_stance_language,chi_square,9405,NaN,0.103948,0.096707,0.115468,4.455678e-58
2,fisher_x_vs_reddit,fisher_exact,5385,0.850869,0.850869,0.688441,1.051620,1.374316e-01
3,mannwhitney_engagement_x_vs_reddit,mann_whitney_u,0,NaN,NaN,NaN,NaN,NaN
4,welch_text_length_x_vs_reddit,welch_t_test,168288,-6.047552,-0.017990,-0.034375,-0.001606,1.407000e-08
5,fisher_x_vs_youtube,fisher_exact,4146,0.525503,0.525503,0.423736,0.651712,3.204855e-09
6,mannwhitney_engagement_x_vs_youtube,mann_whitney_u,0,NaN,NaN,NaN,NaN,NaN
7,welch_text_length_x_vs_youtube,welch_t_test,82359,109.220088,0.538896,0.521354,0.556438,0.000000e+00
8,fisher_reddit_vs_youtube,fisher_exact,5333,0.617607,0.617607,0.514241,0.741751,2.814935e-07
9,mannwhitney_engagement_reddit_vs_youtube,mann_whitney_u,0,NaN,NaN,NaN,NaN,NaN


### تفسیر (فقط برای نمونه‌ی مشاهده‌شده، نه جمعیت کلی)

- **Chi-square (Stance x Platform/Language):** یک Cramer's V کوچک تا متوسط نشان می‌دهد پلتفرم/زبان با توزیع Stance رابطه دارد -- این «همراهی»، نه علیت است.
- **Fisher's Exact (۲×۲):** روی support/oppose بین هر جفت پلتفرم، با Odds Ratio و CI دقیق (نه تقریبی).
- **Mann-Whitney (Engagement):** `engagement_score` برای X/Reddit در این فایل کاملاً خالی است (هیچ Collector آن را پر نمی‌کند) -- سطرهای مربوطه `not_applicable` با دلیل صریح هستند؛ مقایسه‌ی واقعی فقط داخل YouTube (مثبت/منفی) انجام شد.
- **Welch's t (طول متن):** هر سه جفت پلتفرم قابل مقایسه بودند.

---
## بخش ۲ — تحلیل حساسیت (فاز پانزدهم، حداقل ۶ مقایسه)

معیار مشترک همه‌ی مقایسه‌های زیر: سهم `sentiment_label == "positive"` در رکوردهای `annotation_status == "ok"`، زیر یک شرط پایه (Baseline) در برابر یک شرط جایگزین (Alternative).

In [4]:
sens_path = TABLES_DIR / "sensitivity_analysis_results.csv"
if sens_path.exists():
    sensitivity = pd.read_csv(sens_path)
    print(f"{len(sensitivity)} row(s) across {sensitivity['comparison_id'].nunique()} distinct comparison(s)")
    display(sensitivity[[
        "comparison_id", "platform", "n_baseline", "positive_share_baseline",
        "n_alternative", "positive_share_alternative", "shift", "note",
    ]])
else:
    print("STATUS: pending -- run `python -m src.temporal_analysis.sensitivity_analysis` first.")

16 row(s) across 6 distinct comparison(s)


,comparison_id,platform,n_baseline,positive_share_baseline,n_alternative,positive_share_alternative,shift,note
0,duplicate_inclusion,x,2519,0.075824,2401,0.075385,-0.000438,حذف‌شده: 118 رکورد duplicate/near-duplicate
1,duplicate_inclusion,reddit,4486,0.055506,4346,0.056604,0.001098,حذف‌شده: 140 رکورد duplicate/near-duplicate
2,duplicate_inclusion,youtube,2400,0.241667,2323,0.235041,-0.006626,حذف‌شده: 77 رکورد duplicate/near-duplicate
3,content_vs_author_balanced,x,2519,0.075824,1884,0.080117,0.004293,n_alternative = تعداد author_hash یکتا (مخرج A...
4,content_vs_author_balanced,reddit,4486,0.055506,3054,0.061161,0.005655,n_alternative = تعداد author_hash یکتا (مخرج A...
5,content_vs_author_balanced,youtube,2400,0.241667,2077,0.239881,-0.001786,n_alternative = تعداد author_hash یکتا (مخرج A...
6,largest_source_exclusion,x,2519,0.075824,0,NaN,NaN,not_meaningful: source_container یک مقدار ثابت...
7,largest_source_exclusion,reddit,4486,0.055506,3719,0.054585,-0.000921,بزرگ‌ترین Source حذف‌شده: 'r/worldnews' (767 ر...
8,largest_source_exclusion,youtube,2400,0.241667,1793,0.264919,0.023252,بزرگ‌ترین Source حذف‌شده: 'Iran International ...
9,largest_parent_exclusion,x,2519,0.075824,0,NaN,NaN,post_id برای این پلتفرم خالی است -- قابل محاسب...


### سه مقایسه‌ی دیگر (به‌عنوان محصول جانبی اسکریپت‌های دیگر از قبل موجودند، نه دوباره‌محاسبه‌شده)

طبق `outputs/tables/sensitivity_analysis_reference_index.md`:

| مقایسه | فایل |
|---|---|
| Spearman در برابر Pearson | `outputs/tables/financial/financial_social_correlation_results_v1.csv` (ستون `method`) |
| Event window ۱، ۲ و ۳ هفته‌ای | `outputs/tables/event_analysis/event_study_sensitivity_window.csv` |
| هر پلتفرم جدا در برابر Pooled observed | `outputs/tables/descriptive_stats_by_platform_week.csv` در برابر `descriptive_stats_by_week_pooled_all_platforms.csv` |

In [5]:
ref_index_path = TABLES_DIR / "sensitivity_analysis_reference_index.md"
if ref_index_path.exists():
    print(ref_index_path.read_text(encoding="utf-8"))

# تحلیل حساسیت — مقایسه‌های موجود در جاهای دیگر (نه اینجا محاسبه‌شده)

این‌ها به‌عنوان بخشی از اسکریپت‌های دیگر Pipeline B از قبل تولید شده‌اند؛ اینجا فقط رفرنس داده می‌شود.

| مقایسه | فایل |
|---|---|
| Spearman در برابر Pearson (همبستگی مالی) | `outputs/tables/financial/financial_social_correlation_results_v1.csv` (ستون `method`) |
| Event window ۱، ۲ و ۳ هفته‌ای | `outputs/tables/event_analysis/event_study_sensitivity_window.csv` |
| هر پلتفرم جدا در برابر Pooled observed | `outputs/tables/descriptive_stats_by_platform_week.csv` در برابر `descriptive_stats_by_week_pooled_all_platforms.csv` |
| با/بدون بزرگ‌ترین Source/Near-duplicate (رویداد) | `outputs/tables/event_analysis/event_study_sensitivity_robustness.csv` |


---
## بخش ۳ — Claim Registry

خلاصه‌ی ادعاهای مستند این پروژه، هرکدام مستقیماً به یک فایل/سلول خروجی مرتبط. سند کامل: [`docs/claim_registry.md`](../docs/claim_registry.md).

> **اصل بنیادین:** نمونه‌ی این پروژه غیراحتمالی (Non-probability) است. هیچ ادعایی در این Notebook نماینده‌ی جمعیت کلی («مردم ایران»، «مردم آمریکا»، ...) نیست -- فقط توصیف نمونه‌ی مشاهده‌شده روی X/Reddit/YouTube در بازه‌ی ثبت‌شده است. تا وقتی annotation کامل نشده (۳.۷٪ پوشش فعلی)، هر عدد زیر مقدماتی/Exploratory است.

In [6]:
claim_registry_path = ROOT / "docs" / "claim_registry.md"
if claim_registry_path.exists():
    text = claim_registry_path.read_text(encoding="utf-8")
    start = text.find("## بخش ۲")
    end = text.find("## بخش ۳")
    print(text[start:end] if start != -1 else text)
else:
    print("STATUS: pending -- docs/claim_registry.md not found.")

## بخش ۲ — ادعاهای مستند (هرکدوم فقط با پوشش فعلی ۳.۷٪ معتبرند)

| # | ادعا | خروجی/سلول مرجع | محدودیت |
|---|---|---|---|
| C01 | توزیع Stance بین سه پلتفرم تفاوت معنادار آماری دارد (Cramér's V=۰.۱۴۷, p<۰.۰۰۱) | `outputs/tables/group_comparison_results.csv`, ردیف `chi_square_stance_platform` | فقط روی ۹,۴۰۵ رکورد annotate‌شده؛ فقط «همراهی»، نه علیت |
| C02 | طول متن بین X و YouTube تفاوت معنادار دارد (Hedges' g=۰.۵۴، YouTube بلندتر) | `outputs/tables/group_comparison_results.csv`, ردیف `welch_text_length_x_vs_youtube` | Effect size متوسط، نه بزرگ |
| C03 | حذف Duplicate/Near-duplicate سهم Sentiment مثبت را به‌طور معنادار جابه‌جا نمی‌کند (Shift<۱٪ در هر سه پلتفرم) | `outputs/tables/sensitivity_analysis_results.csv`, `comparison_id=duplicate_inclusion` | فقط ۳۳۵ رکورد Near-duplicate در نمونه‌ی annotate‌شده‌ی فعلی |
| C04 | Engagement-weighting (log1p) سهم مثبت یوتیوب را از ۲۴.۲٪ به ۲۹.۱٪ افزایش می‌دهد | `outputs/tables/sensitivity_analysis_results.csv`, `comparison_id=engagement_weight

---
## جمع‌بندی

- هر ۴ آزمون §24 و حداقل ۶ مقایسه‌ی حساسیت فاز پانزدهم روی داده‌ی واقعی (هرچند جزئی) اجرا شدند -- هیچ‌کدام روی داده‌ی Synthetic نیستند.
- هر خروجی این Notebook مستقیماً از یک فایل CSV در `outputs/tables/` می‌آید که با اجرای مجدد اسکریپت مربوطه (نه این Notebook) به‌روزرسانی می‌شود.
- وضعیت کامل ۲۲ خروجی نهایی اجباری چک‌لیست: `docs/claim_registry.md`، بخش ۱.
- Open Items باقی‌مانده (خرابی content_id در Gold Sample، Full Annotation ناقص، Notebookهای ۰۱-۰۴ Pipeline A، گزارش نهایی): `docs/claim_registry.md`، بخش ۳.